# Замыкания и декораторы

**Результат:** Создавать функции с сохранёнными параметрами и обёртки с прежним контрактом.


## 1. Замыкание

Замыкание — функция, использующая переменные окружающей области после завершения внешнего вызова. Так можно настроить правило один раз и передавать готовую функцию как зависимость.

Вложенная функция обращается к переменным внешней функции. Каждый вызов фабрики создаёт своё окружение. Если нужно переприсвоить захваченную переменную, используется nonlocal; без него присваивание создаст локальную переменную. Изменять объект по ссылке и переприсваивать имя — разные действия.


In [ ]:
def make_prefix(prefix):
    def format_text(text):
        return prefix + text
    return format_text

api = make_prefix("API: ")
assert api("ok") == "API: ok"
assert make_prefix("SQL: ")("ok") == "SQL: ok"


**Практика:** Создать два независимых форматтера с разными префиксами.


## 2. Обёртка и @

Декоратор принимает функцию и возвращает заменяющий вызываемый объект. Обычно это обёртка, добавляющая проверку, учёт или подготовку вокруг исходного вызова. Запись @decorate перед def эквивалентна последующему присваиванию f = decorate(f).

Обёртка должна передать *args и **kwargs и вернуть результат исходной функции. Если забыть return, вызывающий получит None. Если без причины перехватить исключение, изменится контракт. wraps переносит имя, docstring и ссылку __wrapped__ на исходную функцию.


In [ ]:
from functools import wraps
def stripped_result(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        return function(*args, **kwargs).strip()
    return wrapper

@stripped_result
def greeting(name):
    """Вернуть приветствие."""
    return f" hello {name} "

assert greeting(name="ira") == "hello ira"
assert greeting.__name__ == "greeting"


**Практика:** Объяснить разницу между @stripped_result и вызовом greeting.


## 3. Декоратор с настройкой

Для настройки нужны два этапа: фабрика принимает параметр, возвращает декоратор, а декоратор получает функцию. Поэтому @add_prefix("! ") сначала создаёт настроенный декоратор.

Не стоит декорировать всё подряд: явный вызов простой функции может быть понятнее. При создании замыканий в цикле переменная захватывается как переменная, а не снимок её значения; отдельная фабрика даёт независимую настройку для каждого результата.


In [ ]:
def add_prefix(prefix):
    def decorate(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            return prefix + function(*args, **kwargs)
        return wrapper
    return decorate

@add_prefix("! ")
def label(text):
    return text.upper()

assert label("api") == "! API"


**Практика:** Назвать значения, принимаемые каждым из трёх уровней функций.


Практические задания находятся в [homework.ipynb](homework.ipynb).


## Справочные источники

- [functools.wraps](https://docs.python.org/3.14/library/functools.html#functools.wraps)
